# GST-registered companies: targeting walkthrough

This notebook walks through the analysis one step at a time, using the processed table in `data/processed/companies.parquet`. The same functions are run by `run_analysis.py`.

In [1]:
import os, sys
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, '.')

import pandas as pd
from src import analysis, config as cfg, industry, prepare, sql_queries
pd.set_option('display.width', 160)

df = prepare.read_processed()
print(f'{len(df):,} private companies in the extract (snapshot {cfg.SNAPSHOT.date()})')
df.head()

4,088,709 private companies in the extract (snapshot 2026-10-21)


,abn_status,abn_status_from,gst_status,gst_status_from,state,postcode,has_business_name,dgr,sector,operating,sa4,sa3
0,CAN,2019-05-01,CAN,2019-05-02,NSW,2141,0,0,Unclassified,1,Sydney - Parramatta,Auburn
1,ACT,1999-11-01,ACT,2000-07-01,NSW,2015,0,0,Unclassified,1,Sydney - City and Inner South,Sydney Inner City
2,CAN,2014-12-18,NON,NaT,NSW,2000,0,0,Manufacturing and engineering,1,Sydney - City and Inner South,Sydney Inner City
3,ACT,2000-05-07,ACT,2000-07-01,NSW,2113,1,0,"Cleaning, security and facilities",1,Sydney - Ryde,Ryde - Hunters Hill
4,CAN,2022-05-20,CAN,2022-05-12,NSW,2010,1,0,Unclassified,1,Sydney - City and Inner South,Sydney Inner City


## 1. Market size

Active companies only. "Leads" are operating companies (not holding, trustee or numbered companies) that are registered for GST.

In [2]:
analysis.market_summary(df).round(1)

,state,active_companies,gst_registered_pct,new_last_12m,new_prior_12m,leads_last_12m,growth_pct,new_per_100_active
0,NSW,826585,67.4,80592,73517,50616,9.6,9.7
1,VIC,556500,66.5,58859,51259,37787,14.8,10.6
2,QLD,399080,66.4,44636,37815,28290,18.0,11.2
3,WA,193266,65.7,22795,19064,14516,19.6,11.8
4,SA,122188,58.4,12851,10826,7746,18.7,10.5
5,TAS,25340,69.7,2152,1887,1467,14.0,8.5
6,ACT,33763,67.0,3522,3057,2236,15.2,10.4
7,NT,12310,71.5,1301,1176,863,10.6,10.6
8,Australia,2169536,66.4,226712,198621,143523,14.1,10.4


## 2. New companies by month

Counted by the month a company's ABN became active. Only companies that are still active are in the extract with that date, so older months are understated (see the README).

In [3]:
monthly = analysis.monthly_new(df)
monthly.tail(12)

,month,new_companies,gst_registered
33,2025-10-01,18279,12853
34,2025-11-01,16360,11316
35,2025-12-01,13715,9469
36,2026-01-01,15298,10592
37,2026-02-01,17767,12319
38,2026-03-01,19725,13506
39,2026-04-01,17755,11885
40,2026-05-01,19837,12986
41,2026-06-01,24299,15471
42,2026-07-01,25161,16569


In [4]:
analysis.seasonality(monthly).round(1)

,month_no,share_of_year_pct,vs_average_month_pct
0,1,6.6,-21.3
1,2,7.5,-10.4
2,3,8.1,-3.2
3,4,7.2,-13.6
4,5,8.9,6.5
5,6,10.1,21.6
6,7,11.3,35.7
7,8,8.7,4.3
8,9,8.3,0.1
9,10,8.7,4.6


## 3. How soon do companies register for GST?

Companies started January 2023 to September 2025, so each has had at least a year.

In [5]:
curve, stats = analysis.gst_timing(df)
print({k: round(float(v), 1) for k, v in stats.items()})
curve.round(1)

{'companies': 483781.0, 'gst_registered': 348855.0, 'gst_registered_pct': 72.1, 'same_day_pct': 75.9, 'within_30_days_pct': 84.9, 'within_90_days_pct': 89.7, 'backdated_before_abn_pct': 0.4}


,days_after_abn,share_of_gst_registered_pct
0,0,76.3
1,7,80.2
2,30,84.9
3,60,87.9
4,90,89.7
5,180,92.9
6,365,96.2


## 4. Industry from names

The extract has no industry code, so `src/industry.py` reads the legal and business names. A few examples:

In [6]:
examples = [('SMITH PLUMBING PTY LTD', ''), ('SMITH FAMILY HOLDINGS PTY LTD', 'Smith Plumbing'),
            ('XYZ PROPERTY HOLDINGS PTY LTD', ''), ('BLAFIN PTY LTD', '')]
for name, business in examples:
    print(f'{name:32} {business:16} -> {industry.classify_one(name, business)}')

SMITH PLUMBING PTY LTD                            -> Construction and trades
SMITH FAMILY HOLDINGS PTY LTD    Smith Plumbing   -> Construction and trades
XYZ PROPERTY HOLDINGS PTY LTD                     -> Holding, investment or trustee
BLAFIN PTY LTD                                    -> Unclassified


In [7]:
check = pd.read_csv('outputs/classifier_check_sample.csv')
print(f"Hand-checked sample: {(check['correct'] == 'Y').sum()} of {len(check)} correct")
check[check['correct'] == 'N'][['name', 'business_names', 'assigned_sector', 'note']]

Hand-checked sample: 189 of 200 correct


,name,business_names,assigned_sector,note
35,Building Careers Pty Ltd,CONSTRUCTION CAREERS,Construction and trades,"Recruitment, not construction"
44,CV WELDING MINING MAINTENANCE PTY LTD,NaN,"Cleaning, security and facilities","Welding/mining maintenance, not cleaning"
49,RASEAL AGENCIES PTY. LTD.,LIN-AIR HEATING & AIR CONDITIONING,Retail and wholesale,Heating and air conditioning installer
74,SLADE TRADING PTY LTD,North Coast Haulage,Retail and wholesale,Trades as a haulage business
78,CAPITAL BAY PTY LTD,NaN,"Holding, investment or trustee",Name gives no real clue; 'Capital' is not enough
98,EZY MEDIA GROUP PTY LTD,Benchmark Care | Optimum Plan Management | Pro...,"Media, creative and events","Care and plan management, not media"
100,SMART RESOURCES QLD PTY LTD,Punjabi Palace,Mining and energy,Trades as a restaurant
112,MELCA NOMINEES PTY LTD,Melbourne Carpet Specialists | Melbourne Carpe...,"Holding, investment or trustee",Trades as a carpet retailer
147,STEELBAR RESTORATION PTY LTD,STEELBAR MECHANICAL & ROADWORTHY'S | STEELBAR ...,Transport and logistics,"Mechanical repairs, so automotive"
175,SKELLYWAGG TREES PTY LTD,KOOTENAY PARK NURSERY,Health and care,"Plant nursery, not nursing"


In [8]:
analysis.sector_summary(df).round(1)

,sector,active_companies,gst_registered_pct,new_last_12m,new_prior_12m,growth_pct,share_of_active_pct
0,Unclassified,1022732,62.7,112026,97365,15.1,47.1
1,"Holding, investment or trustee",205270,35.0,20193,15800,27.8,9.5
2,Construction and trades,193061,84.2,19458,17553,10.9,8.9
3,Health and care,83576,76.6,9566,8747,9.4,3.9
4,Professional services,114414,74.2,9521,8888,7.1,5.3
5,Property and real estate,77482,65.5,7501,6265,19.7,3.6
6,IT and digital,56084,73.9,6793,5186,31.0,2.6
7,Transport and logistics,52358,84.5,6316,5222,20.9,2.4
8,Food and hospitality,61018,81.2,5924,6061,-2.3,2.8
9,Retail and wholesale,54307,74.7,5057,4967,1.8,2.5


## 5. Victoria by region

In [9]:
analysis.region_summary(df).round(1)

,sa4,active_companies,new_last_12m,new_prior_12m,leads_last_12m,growth_pct,new_per_100_active
0,Melbourne - West,76816,11557,9452,7926,22.3,15.0
1,Melbourne - South East,75885,8881,7946,5888,11.8,11.7
2,Melbourne - Inner,93965,8565,7490,4931,14.4,9.1
3,Melbourne - North West,43806,5485,4653,3657,17.9,12.5
4,Melbourne - Inner South,49417,4099,3498,2362,17.2,8.3
5,Melbourne - North East,34680,4077,3476,2651,17.3,11.8
6,Melbourne - Inner East,38063,3331,3225,1897,3.3,8.8
7,Melbourne - Outer East,36183,3193,2880,2016,10.9,8.8
8,Mornington Peninsula,27689,2708,2381,1821,13.7,9.8
9,Geelong,18990,1892,1690,1183,12.0,10.0


## 6. Target segments

Region x industry segments with at least 100 new leads, split at the median size and the median growth.

In [10]:
seg = analysis.segments(df)
print(seg.attrs)
seg.round(1).head(15)

{'size_cut': 168.0, 'growth_cut': 4.253756260434061}


,rank,sa4,sector,new_leads,prior_leads,growth_pct,quadrant
0,1,Melbourne - West,Construction and trades,851,729.0,16.7,Priority: large and growing
1,2,Melbourne - West,Transport and logistics,684,491.0,39.3,Priority: large and growing
2,3,Melbourne - South East,Construction and trades,626,599.0,4.5,Priority: large and growing
3,4,Melbourne - North West,Construction and trades,514,429.0,19.8,Priority: large and growing
4,5,Melbourne - Inner,Professional services,377,343.0,9.9,Priority: large and growing
5,6,Melbourne - North East,Construction and trades,350,309.0,13.3,Priority: large and growing
6,7,Melbourne - South East,Transport and logistics,287,244.0,17.6,Priority: large and growing
7,8,Melbourne - South East,Professional services,245,224.0,9.4,Priority: large and growing
8,9,Melbourne - West,IT and digital,239,192.0,24.5,Priority: large and growing
9,10,Melbourne - North West,Transport and logistics,216,157.0,37.6,Priority: large and growing


## 7. The same results in SQL

The queries in `sql/` run with DuckDB directly on the Parquet file. The tests check that they match the pandas results.

In [11]:
sql_queries.run('04_vic_segments').round(1).head(10)

,rank,sa4,sector,new_leads,prior_leads,growth_pct,quadrant
0,1,Melbourne - West,Construction and trades,851,729,16.7,Priority: large and growing
1,2,Melbourne - West,Transport and logistics,684,491,39.3,Priority: large and growing
2,3,Melbourne - South East,Construction and trades,626,599,4.5,Priority: large and growing
3,4,Melbourne - North West,Construction and trades,514,429,19.8,Priority: large and growing
4,5,Melbourne - Inner,Professional services,377,343,9.9,Priority: large and growing
5,6,Melbourne - North East,Construction and trades,350,309,13.3,Priority: large and growing
6,7,Melbourne - South East,Transport and logistics,287,244,17.6,Priority: large and growing
7,8,Melbourne - South East,Professional services,245,224,9.4,Priority: large and growing
8,9,Melbourne - West,IT and digital,239,192,24.5,Priority: large and growing
9,10,Melbourne - North West,Transport and logistics,216,157,37.6,Priority: large and growing
